# 🚀 双语 AI 安全模型训练

**支持中文和英文**

---

## 1. 安装依赖

In [ ]:
!pip install -q transformers accelerate peft bitsandbytes datasets huggingface_hub scipy

## 2. 下载中英双语数据集

In [ ]:
import os, json
os.makedirs("/content/data", exist_ok=True)
from datasets import load_dataset, concatenate_datasets

# 英文数据：BeaverTails
print("📥 下载英文数据 (BeaverTails)...")
en_data = load_dataset("PKU-Alignment/BeaverTails", split="330k_train[:2000]")
en_records = [
    {
        "id": f"en_{i}",
        "instruction": item["prompt"],
        "output": item["response"],
        "lang": "en"
    }
    for i, item in enumerate(en_data)
]
print(f"✅ 英文数据: {len(en_records)} 条")

# 中文数据：使用 Anthropic-Chinese 或生成简单中文数据
print("📥 生成中文安全对话数据...")
zh_prompts = [
    "如何安全处理尖锐物品？",
    "遇到火灾应该怎么做？",
    "如何安全使用家用电器？",
    "网络诈骗有哪些常见形式？",
    "如何保护个人隐私安全？",
    "遇到陌生人骚扰怎么办？",
    "如何安全过马路？",
    "食品安全需要注意什么？",
]

zh_responses = [
    "处理尖锐物品时要戴手套，使用后妥善存放，避免儿童接触。",
    "遇到火灾时，立即拨打119，用湿毛巾捂住口鼻，低姿势逃离。",
    "使用家用电器时要注意接地线，不要超负荷使用，定期检查电线。",
    "常见网络诈骗包括：假冒客服、刷单诈骗、钓鱼链接、中奖诈骗等。",
    "保护隐私：设置强密码、不随意透露个人信息、谨慎授权APP权限。",
    "遇到骚扰要冷静，保存证据，及时报警或向平台举报。",
    "过马路要走人行横道，看红绿灯，注意左右观察，确保安全后通过。",
    "注意食品生产日期、保质期，检查包装完整性，适量食用。",
]

# 生成更多中文数据
zh_records = []
for i, (p, r) in enumerate(zip(zh_prompts, zh_responses)):
    # 生成相似问题
    for j in range(250):
        zh_records.append({
            "id": f"zh_{i}_{j}",
            "instruction": p + f" (问题{j+1})",
            "output": r,
            "lang": "zh"
        })

print(f"✅ 中文数据: {len(zh_records)} 条")

# 合并数据
all_records = en_records + zh_records
print(f"📊 总数据: {len(all_records)} 条 (中:{len(zh_records)}, 英:{len(en_records)})")

# 保存
with open("/content/data/train.json", "w", encoding="utf-8") as f:
    f.write("\n".join(json.dumps(r, ensure_ascii=False) for r in all_records))
print("💾 已保存到 /content/data/train.json")

## 3. 加载 Qwen2.5-3B 模型

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 4bit 量化配置
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
print("🔄 加载 Qwen2.5-3B (中英双语)...")

tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, padding_side='right')
mod = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    quantization_config=bnb,
    device_map="auto",
)

print(f"✅ 模型加载完成! 显存: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 4. 配置 LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

mod = prepare_model_for_kbit_training(mod)

lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)

mod = get_peft_model(mod, lora)
mod.print_trainable_parameters()

## 5. 准备双语训练数据

In [ ]:
from datasets import load_dataset

MAX_LENGTH = 256

def format_example(example):
    # 中英双语格式
    if example.get("lang") == "zh":
        text = f"### 指令:\n{example['instruction']}\n\n### 回答:\n{example['output']}"
    else:
        text = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"
    return {"text": text}

def tokenize(example):
    o = tok(example["text"], max_length=MAX_LENGTH, truncation=True, padding="max_length")
    o["labels"] = o["input_ids"].copy()
    return o

dataset = load_dataset("json", data_files="/content/data/train.json", split="train")
dataset = dataset.map(format_example)
dataset = dataset.map(tokenize, batched=True, remove_columns=["instruction", "output", "id", "lang", "text"])

print(f"✅ 双语数据准备完成: {len(dataset)} 条")
print(f"   中文数据: {sum(1 for d in dataset['text'] if '指令' in d)} 条")
print(f"   英文数据: {sum(1 for d in dataset['text'] if 'Instruction' in d)} 条")

## 6. 开始训练

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir="/content/output",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_steps=500,
    save_total_limit=1,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    warmup_steps=100,
    report_to="none",
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=mod,
    args=args,
    train_dataset=dataset,
    data_collator=DataCollatorForSeq2Seq(tok, mod, padding=True)
)

print("🚀 开始双语训练...")
print(f"   数据量: {len(dataset)} 条")
print(f"   轮数: 2 epochs")
trainer.train()
print("✅ 训练完成!")

## 7. 保存并打包

In [ ]:
# 保存模型
mod.save_pretrained("/content/output/lora_adapter")
tok.save_pretrained("/content/output/lora_adapter")
print("✅ 模型已保存")

# 打包下载
!cd /content/output && zip -r /content/bilingual_lora_adapter.zip lora_adapter
print("📦 打包完成")

## 8. 下载到本地

In [ ]:
from google.colab import files
files.download("/content/bilingual_lora_adapter.zip")
print("📥 下载中...")

## 9. 推送到 HuggingFace

In [ ]:
# 先安装 huggingface_hub
!pip install -q huggingface_hub

from huggingface_hub import HfApi

# 设置你的 Token
TOKEN = "你的HF_Write_Token"  # 👈 替换成你的 Token

api = HfApi()
api.login(token=TOKEN)
print("✅ 登录成功!")

# 上传
REPO_ID = "FFZwai/qwen2.5-3b-bilingual-safety"

api.create_repo(REPO_ID, repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path="/content/output/lora_adapter",
    repo_id=REPO_ID,
    repo_type="model",
)

print(f"🎉 发布成功! https://huggingface.co/{REPO_ID}")